<a href="https://colab.research.google.com/github/lopezreyesleonardoa-sketch/Representaci-n-del-conocimiento/blob/main/PIA_EDOO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Leonardo Alfredo López Reyes 2181755

#Sistema de apoyo para la distribución de medicamentos en unidades médicas del IMSS

###Importación de librerías y datos de entrada

In [14]:
import heapq

# Inventario inicial
inventario = [
    {"codigo": "M-001", "nombre": "Medicamento A", "existencias": 50},
    {"codigo": "M-002", "nombre": "Medicamento B", "existencias": 30},
    {"codigo": "M-003", "nombre": "Medicamento C", "existencias": 20}
]

# Solicitudes recibidas
solicitudes = [
    {
        "id": "S-001",
        "cantidad": 20,
        "valoracion_completa": True,
        "urgencia_inmediata": True,
        "urgencia_proxima": False,
        "beneficio_valorado": True,
        "prioridad": "Alta"
    },
    {
        "id": "S-002",
        "cantidad": 30,
        "valoracion_completa": True,
        "urgencia_inmediata": False,
        "urgencia_proxima": True,
        "beneficio_valorado": True,
        "prioridad": "Media"
    },
    {
        "id": "S-003",
        "cantidad": 25,
        "valoracion_completa": False,
        "urgencia_inmediata": True,
        "urgencia_proxima": False,
        "beneficio_valorado": False,
        "prioridad": None
    }
]

###Implementación del árbol de decisión

In [16]:
# Clase para representar los nodos del arbol
class Nodo:
    def __init__(self, pregunta, si=None, no=None, resultado=None):
        self.pregunta = pregunta
        self.si = si
        self.no = no
        self.resultado = resultado


# Funcion para construir el arbol
def crear_arbol():

    pendiente = Nodo(
        "Pendiente de valoracion medica",
        resultado="Pendiente de valoracion"
    )

    revision = Nodo(
        "Revision medica complementaria",
        resultado="Requiere revision"
    )

    cola_alta = Nodo(
        "Pasar a cola de prioridad",
        resultado="Continua al proceso logistico"
    )

    cola_otra = Nodo(
        "Pasar a cola de prioridad",
        resultado="Continua al proceso logistico"
    )

    seguimiento = Nodo(
        "Seguimiento de la solicitud",
        resultado="Mantener en seguimiento"
    )

    beneficio = Nodo(
        "¿El beneficio terapeutico fue valorado?",
        si=cola_alta,
        no=revision
    )

    urgencia_proxima = Nodo(
        "¿La urgencia es proxima o programada?",
        si=cola_otra,
        no=seguimiento
    )

    urgencia = Nodo(
        "¿Existe urgencia inmediata?",
        si=beneficio,
        no=urgencia_proxima
    )

    raiz = Nodo(
        "¿La valoracion medica esta completa?",
        si=urgencia,
        no=pendiente
    )

    return raiz


# Funcion para evaluar cada solicitud
def evaluar_solicitud(raiz, solicitud):

    nodo = raiz

    while nodo.resultado is None:

        pregunta = nodo.pregunta

        if "valoracion medica" in pregunta:
            condicion = solicitud["valoracion_completa"]

        elif "urgencia inmediata" in pregunta:
            condicion = solicitud["urgencia_inmediata"]

        elif "beneficio terapeutico" in pregunta:
            condicion = solicitud["beneficio_valorado"]

        elif "proxima o programada" in pregunta:
            condicion = solicitud["urgencia_proxima"]

        nodo = nodo.si if condicion else nodo.no

    return nodo.resultado


# Construccion y recorrido del arbol
arbol = crear_arbol()

print("=== ARBOL DE DECISION ===")

solicitudes_autorizadas = []

for solicitud in solicitudes:

    resultado = evaluar_solicitud(arbol, solicitud)

    print(solicitud["id"], ":", resultado)

    if resultado == "Continua al proceso logistico":
        solicitudes_autorizadas.append(solicitud)

=== ARBOL DE DECISION ===
S-001 : Continua al proceso logistico
S-002 : Continua al proceso logistico
S-003 : Pendiente de valoracion


###Implementación de la tabla hash

In [22]:
# Clase para implementar la tabla hash
class TablaHash:

    def __init__(self, tamaño=5):
        self.tamaño = tamaño
        self.tabla = [[] for _ in range(tamaño)]

    # Funcion hash
    def funcion_hash(self, codigo):
        clave = int(codigo.split("-")[1]) + 100
        return clave % self.tamaño

    # Insertar medicamento
    def insertar(self, codigo, nombre, existencias):

        posicion = self.funcion_hash(codigo)

        for elemento in self.tabla[posicion]:

            if elemento[0] == codigo:
                elemento[1] = nombre
                elemento[2] = existencias
                return

        self.tabla[posicion].append(
            [codigo, nombre, existencias]
        )

    # Buscar medicamento
    def buscar(self, codigo):

        posicion = self.funcion_hash(codigo)

        for elemento in self.tabla[posicion]:

            if elemento[0] == codigo:
                return elemento

        return None


# Crear tabla hash
tabla_hash = TablaHash()

# Registrar medicamentos
for medicamento in inventario:

    tabla_hash.insertar(
        medicamento["codigo"],
        medicamento["nombre"],
        medicamento["existencias"]
    )


# Consultar medicamento
print("=== TABLA HASH ===")

medicamento = tabla_hash.buscar("M-001")

if medicamento:

    print("Codigo:", medicamento[0])
    print("Nombre:", medicamento[1])
    print("Existencias:", medicamento[2])

=== TABLA HASH ===
Codigo: M-001
Nombre: Medicamento A
Existencias: 50


###Implementación de la cola de prioridad

In [21]:
# Clase para implementar la cola de prioridad
class ColaPrioridad:

    def __init__(self):
        self.cola = []
        self.contador = 0

    # Insertar solicitud
    def insertar(self, solicitud):

        prioridades = {
            "Alta": 1,
            "Media": 2,
            "Baja": 3
        }

        prioridad = prioridades[solicitud["prioridad"]]

        heapq.heappush(
            self.cola,
            (prioridad, self.contador, solicitud)
        )

        self.contador += 1

    # Retirar la siguiente solicitud
    def atender(self):

        if self.cola:
            elemento = heapq.heappop(self.cola)
            return elemento[2]

        return None


# Crear cola de prioridad
cola = ColaPrioridad()

# Insertar solamente solicitudes autorizadas
for solicitud in solicitudes_autorizadas:
    cola.insertar(solicitud)


# Mostrar el orden de las solicitudes
print("=== COLA DE PRIORIDAD ===")

while cola.cola:

    solicitud = cola.atender()

    print(
        solicitud["id"],
        "- Prioridad:",
        solicitud["prioridad"]
    )

=== COLA DE PRIORIDAD ===
S-001 - Prioridad: Alta
S-002 - Prioridad: Media


###Análisis de existencias y demanda

In [20]:
# Calcular la cantidad total solicitada
total_solicitado = sum(
    solicitud["cantidad"] for solicitud in solicitudes
)

# Consultar las existencias del medicamento
medicamento = tabla_hash.buscar("M-001")

if medicamento:

    existencias = medicamento[2]

    # Calcular la cantidad faltante
    faltante = max(0, total_solicitado - existencias)

    print("=== ANALISIS DE EXISTENCIAS ===")
    print("Total solicitado:", total_solicitado)
    print("Existencias disponibles:", existencias)
    print("Faltante:", faltante)

=== ANALISIS DE EXISTENCIAS ===
Total solicitado: 75
Existencias disponibles: 50
Faltante: 25
